# 03 — KGML-ag-Carbon Model Training

Builds and trains a **Knowledge-Guided Machine Learning** GRU-based model adapted from:

> Liu et al. (2024). Knowledge-guided machine learning can improve carbon cycle quantification in agroecosystems. *Nature Communications* 15:357.

**Architecture:** 2-layer GRU (64 hidden units) with 3 output heads (NEE, Rh, Yield) and knowledge-guided loss functions enforcing:
1. Mass balance: `ΔSOC = -NEE - Yield`
2. Monotonicity: Rh increases with temperature
3. Non-negativity: daily yield ≥ 0

**Training:** 50 epochs on 12K IPCC-derived synthetic samples, CPU-only.

In [ ]:
import numpy as np
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

# Load synthetic data
data = np.load('data/synthetic_train.npz')
features = data['features']        # (N, 365, 8)
crop_ids = data['crop_ids']        # (N,)
delta_soc = data['delta_soc']      # (N,)
daily_nee = data['daily_nee']      # (N, 365)
train_idx = data['train_idx']
val_idx = data['val_idx']
feat_mean = data['feat_mean']
feat_std = data['feat_std']

print(f'Features: {features.shape}  |  Crops: {crop_ids.shape}  |  Targets: {delta_soc.shape}')
print(f'Train: {len(train_idx)}  |  Val: {len(val_idx)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# KGML-SOC Model — GRU with Knowledge-Guided Architecture
# ══════════════════════════════════════════════════════════════════════

class KGMLSocModel(nn.Module):
    """
    Knowledge-Guided Machine Learning model for SOC estimation.
    Adapted from Liu et al. (2024) KGML-ag-Carbon.
    
    Architecture:
      CropEmbedding(8, 4) → InputProjection(12, 32) → GRU(32, 64, 2 layers)
      → 3 output heads: NEE, Rh, Yield
      → Mass balance: ΔSOC = -sum(NEE) - sum(Yield)
    """
    def __init__(self, input_dim=8, crop_vocab=7, crop_embed_dim=4,
                 hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        
        self.crop_embed = nn.Embedding(crop_vocab, crop_embed_dim)
        self.input_proj = nn.Linear(input_dim + crop_embed_dim, hidden_dim // 2)
        
        self.gru = nn.GRU(
            input_size=hidden_dim // 2,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        
        # Three output heads (Liu et al. multi-task architecture)
        self.nee_head = nn.Linear(hidden_dim, 1)    # daily NEE
        self.rh_head = nn.Linear(hidden_dim, 1)     # daily Rh
        self.yield_head = nn.Linear(hidden_dim, 1)  # daily yield proxy
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, features, crop_ids):
        """
        Args:
            features: (batch, 365, 8) — daily weather + soil features
            crop_ids: (batch,) — integer crop type IDs
        Returns:
            dict with keys: nee, rh, yield_proxy, delta_soc (all tensors)
        """
        B, T, _ = features.shape
        
        # Crop embedding → broadcast to all timesteps
        crop_emb = self.crop_embed(crop_ids)              # (B, 4)
        crop_emb = crop_emb.unsqueeze(1).expand(-1, T, -1)  # (B, T, 4)
        
        # Concatenate features + crop embedding
        x = torch.cat([features, crop_emb], dim=-1)       # (B, T, 12)
        x = torch.relu(self.input_proj(x))                # (B, T, 32)
        
        # GRU temporal processing
        h, _ = self.gru(x)                                # (B, T, 64)
        h = self.dropout(h)
        
        # Output heads
        nee = self.nee_head(h).squeeze(-1)                # (B, T)
        rh = self.rh_head(h).squeeze(-1)                  # (B, T)
        yield_proxy = self.yield_head(h).squeeze(-1)      # (B, T)
        
        # Mass balance: ΔSOC = -sum(NEE) - sum(Yield)
        annual_nee = nee.sum(dim=1)                       # (B,)
        annual_yield = yield_proxy.sum(dim=1)             # (B,)
        delta_soc = -annual_nee - annual_yield            # (B,)
        
        return {
            'nee': nee,              # (B, 365)
            'rh': rh,               # (B, 365)
            'yield_proxy': yield_proxy,  # (B, 365)
            'delta_soc': delta_soc,  # (B,)
            'annual_nee': annual_nee,
            'annual_yield': annual_yield,
        }

model = KGMLSocModel()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}  ({n_params/1000:.1f}K)')
print(model)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Knowledge-Guided Loss Function (4 components)
# ══════════════════════════════════════════════════════════════════════

def kgml_loss(outputs, targets, features, w_mass=0.1, w_mono=0.05, w_yield=0.1):
    """
    Knowledge-guided loss function inspired by Liu et al. (2024).
    
    Components:
      L_data:  MSE on predicted vs target delta_SOC
      L_mass:  Mass balance self-consistency (delta_soc should equal -NEE - Yield)
      L_mono:  Rh should increase with temperature (monotonicity constraint)
      L_yield: Daily yield should be non-negative
    """
    pred_dsoc = outputs['delta_soc']
    target_dsoc = targets
    
    # 1. Data loss: MSE on annual delta_SOC
    l_data = nn.functional.mse_loss(pred_dsoc, target_dsoc)
    
    # 2. Mass balance: delta_soc should equal -NEE - Yield (self-consistency)
    mass_balance = -outputs['annual_nee'] - outputs['annual_yield']
    l_mass = nn.functional.mse_loss(pred_dsoc, mass_balance)
    
    # 3. Monotonicity: Rh should increase with temperature
    # Penalize negative correlation between Rh and temperature
    temp = features[:, :, 0]  # temperature is first feature
    rh = outputs['rh']
    # Compute finite differences
    d_temp = temp[:, 1:] - temp[:, :-1]
    d_rh = rh[:, 1:] - rh[:, :-1]
    # Penalize when temp increases but Rh decreases
    violation = torch.relu(-d_rh * torch.sign(d_temp))
    l_mono = violation.mean()
    
    # 4. Yield non-negativity
    l_yield = torch.relu(-outputs['yield_proxy']).mean()
    
    total = l_data + w_mass * l_mass + w_mono * l_mono + w_yield * l_yield
    
    return total, {
        'total': total.item(),
        'data': l_data.item(),
        'mass': l_mass.item(),
        'mono': l_mono.item(),
        'yield': l_yield.item(),
    }

print('Loss function defined with 4 knowledge-guided components')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Training Loop
# ══════════════════════════════════════════════════════════════════════

# Create data loaders
X_train = torch.tensor(features[train_idx])
C_train = torch.tensor(crop_ids[train_idx])
Y_train = torch.tensor(delta_soc[train_idx])

X_val = torch.tensor(features[val_idx])
C_val = torch.tensor(crop_ids[val_idx])
Y_val = torch.tensor(delta_soc[val_idx])

train_ds = TensorDataset(X_train, C_train, Y_train)
val_ds = TensorDataset(X_val, C_val, Y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

# Optimizer + scheduler
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

N_EPOCHS = 50
history = {'train_loss': [], 'val_loss': [], 'l_data': [], 'l_mass': [], 'l_mono': [], 'l_yield': []}

print(f'Training for {N_EPOCHS} epochs  |  Batch size: 64  |  LR: 1e-3')
print(f'Training samples: {len(train_ds)}  |  Val samples: {len(val_ds)}')
print('='*70)

best_val_loss = float('inf')
for epoch in range(N_EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────
    model.train()
    epoch_losses = []
    epoch_components = {'data': [], 'mass': [], 'mono': [], 'yield': []}
    
    for xb, cb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb, cb)
        loss, components = kgml_loss(outputs, yb, xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_losses.append(components['total'])
        for k in epoch_components:
            epoch_components[k].append(components[k])
    
    train_loss = np.mean(epoch_losses)
    
    # ── Validate ──────────────────────────────────────────────────────
    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, cb, yb in val_loader:
            outputs = model(xb, cb)
            loss, components = kgml_loss(outputs, yb, xb)
            val_losses.append(components['total'])
    
    val_loss = np.mean(val_losses)
    scheduler.step(val_loss)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    for k in epoch_components:
        history[f'l_{k}'].append(np.mean(epoch_components[k]))
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../backend/app/models/kgml_soc_model.pt')
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  '
              f'train={train_loss:.6f}  val={val_loss:.6f}  '
              f'L_data={np.mean(epoch_components["data"]):.6f}  '
              f'L_mass={np.mean(epoch_components["mass"]):.6f}  '
              f'lr={lr:.1e}')

print(f'\nBest val loss: {best_val_loss:.6f}')
print(f'Model saved: ../backend/app/models/kgml_soc_model.pt')

In [ ]:
# ── Save model config for backend ─────────────────────────────────────

with open('data/training_config.json') as f:
    training_config = json.load(f)

model_config = {
    'input_dim': 8,
    'crop_vocab': 7,
    'crop_embed_dim': 4,
    'hidden_dim': 64,
    'num_layers': 2,
    'dropout': 0.1,
    'version': '0.1.0',
    'crop_to_id': training_config['crop_to_id'],
    'feat_mean': training_config['feat_mean'],
    'feat_std': training_config['feat_std'],
    'feature_names': training_config['feature_names'],
    'best_val_loss': float(best_val_loss),
    'n_epochs': N_EPOCHS,
    'n_train': training_config['n_train'],
}

with open('../backend/app/models/kgml_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print('Saved: ../backend/app/models/kgml_config.json')
print(json.dumps(model_config, indent=2))

In [ ]:
# ── Training curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].set_yscale('log')

# Loss components
for key in ['l_data', 'l_mass', 'l_mono', 'l_yield']:
    axes[1].plot(history[key], label=key.replace('l_', 'L_'))
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Knowledge-Guided Loss Components')
axes[1].legend(fontsize=8)
axes[1].set_yscale('log')

# Prediction scatter (val set)
model.eval()
with torch.no_grad():
    preds = model(X_val, C_val)['delta_soc'].numpy()
    targets = Y_val.numpy()

axes[2].scatter(targets, preds, alpha=0.3, s=10)
lims = [min(targets.min(), preds.min()), max(targets.max(), preds.max())]
axes[2].plot(lims, lims, 'r--', linewidth=2)
axes[2].set_xlabel('IPCC Target (t C/ha/yr)')
axes[2].set_ylabel('KGML Predicted (t C/ha/yr)')
r2 = 1 - np.sum((preds - targets)**2) / np.sum((targets - targets.mean())**2)
rmse = np.sqrt(np.mean((preds - targets)**2))
axes[2].set_title(f'KGML vs IPCC  |  R²={r2:.4f}  RMSE={rmse:.4f}')

plt.tight_layout()
plt.savefig('data/kgml_training_results.png', dpi=150)
plt.show()
print(f'R² = {r2:.4f}  |  RMSE = {rmse:.4f} t C/ha/yr')